# IRWA Final Project - Part 3: Ranking & Filtering

Farith Godoy (U214405) · Anna Vasileva (U255290)

Pipeline: query -> AND search -> ranking. The AND search gives the same products to every method, only the order changes.

Run all the cells in order from `project_progress/part_3/`. The dataset has to be in `data/` at the root of the repo.

### Setup: required packages

Only needed the first time (restart the kernel afterwards). `gensim` is new in this part (word2vec).

In [ ]:
# Install the required packages (only needed the first time; restart the kernel afterwards)
%pip install nltk pandas numpy matplotlib gensim

## 0. Reused from Parts 1 and 2

### 0.1 Data loading and preprocessing

Same preprocessing as Part 2: lowercase, alphanumeric tokens, stop-word removal, Porter stemming.

**New:** our score (1.3) needs the numerical fields. In the JSON they are strings (`"3.9"`, `"69% off"`, `"1,169"`) and an empty string when missing, so `parse_number` converts them to float (or `None`).

In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import math
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 70)

nltk.download("stopwords", quiet=True, raise_on_error=True)
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

ROOT_DIR = Path.cwd().parent.parent
DATA_PATH = ROOT_DIR / "data" / "fashion_products_dataset.json"
VALIDATION_PATH = ROOT_DIR / "data" / "validation_labels.csv"


# Compound words whose first part is a single letter (it would be lost as a stopword or means nothing alone):
# "t-shirt", "t shirt" and "tshirt" all become "tshirt"
COMPOUNDS = [("t", "shirt"), ("v", "neck"), ("o", "neck"), ("u", "neck"), ("a", "line"), ("x", "large")]
COMPOUND_PATTERN = re.compile(r"\b(" + "|".join(rf"{a}[\s-]?{b}" for a, b in COMPOUNDS) + r")")


# Same preprocessing as Part 2 + compound joining: lowercase, alphanumeric tokens, stopword removal, Porter stemming
def preprocess_text(text):
    if not text:
        return []

    # Join the compounds (remove the hyphen or space) before splitting into tokens
    text = COMPOUND_PATTERN.sub(lambda m: re.sub(r"[\s-]", "", m.group()), text.lower())
    tokens = re.findall(r"[a-z0-9]+", text)
    return [stemmer.stem(t) for t in tokens if t not in stop_words]


# Flatten the list of {key: value} product details into one string (Part 1)
def product_details_to_text(product_details):
    return " ".join(f"{k} {v}" for detail in product_details for k, v in detail.items() if v is not None)


# Convert a numeric string ("3.9", "1,169", "69% off") to float; None if it is empty
def parse_number(value):
    match = re.search(r"\d+(?:\.\d+)?", str(value or "").replace(",", ""))
    return float(match.group()) if match else None


# Load the dataset, add the tokens of each text field and the parsed numerical fields
def load_and_preprocess(path=DATA_PATH):
    with open(path, "r", encoding="utf-8") as f:
        products = json.load(f)

    for p in products:
        # Text fields (same as Part 2)
        p["title_tokens"] = preprocess_text(p["title"])
        p["description_tokens"] = preprocess_text(p["description"])
        p["product_details_tokens"] = preprocess_text(product_details_to_text(p["product_details"]))

        # Numerical fields for our score (out_of_stock is already a bool)
        p["rating"] = parse_number(p["average_rating"])
        p["discount_pct"] = parse_number(p["discount"])
        p["price"] = parse_number(p["selling_price"])
        p["mrp"] = parse_number(p["actual_price"])

    return products


products = load_and_preprocess()
products_by_pid = {p["pid"]: p for p in products}
titles = {p["pid"]: p["title"] for p in products}
print(f"Products loaded and preprocessed: {len(products):,}")

Products loaded and preprocessed: 28,080


### 0.2 Inverted index and AND search

This is the **filtering** step: a product is returned only if it contains all the query terms. The statement asks for conjunctive (AND) queries only, so every ranking method below orders exactly this set of products.

In [3]:
# Index terms of a product: title + description + product details tokens
def get_terms(product):
    return product["title_tokens"] + product["description_tokens"] + product["product_details_tokens"]


# Build the inverted index: term -> list of pids that contain the term
def create_index(products):
    index = defaultdict(list)
    for p in products:
        for term in set(get_terms(p)):
            index[term].append(p["pid"])
    return index


# AND search: return the set of pids that contain ALL the query terms
def search_and(query, index):
    terms = preprocess_text(query)
    if not terms:
        return set()

    # Intersect the postings lists from the shortest to the longest; stop if it becomes empty
    postings = sorted((set(index.get(term, [])) for term in set(terms)), key=len)
    docs = postings[0]
    for posting in postings[1:]:
        docs &= posting
        if not docs:
            break
    return docs


index = create_index(products)
N = len(products)

# The 5 test queries of Part 2 and the 2 validation queries
test_queries = [
    "men black polo shirt",
    "women printed cotton kurta",
    "grey track pants",
    "men denim jacket",
    "women yoga pants",
]
validation_queries = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}

for query in test_queries:
    print(f"{query!r:32} -> {len(search_and(query, index))} AND results")

'men black polo shirt'           -> 29 AND results
'women printed cotton kurta'     -> 42 AND results
'grey track pants'               -> 327 AND results
'men denim jacket'               -> 73 AND results
'women yoga pants'               -> 19 AND results


## 1. Ranking methods

Every ranking function receives a query, runs `search_and` and returns a list of `(pid, score)` sorted from the highest to the lowest score.

### 1.1 a) TF-IDF + cosine similarity (reused from Part 2)

- $tf_{t,d} = f_{t,d} / |d|$, $\quad idf_t = \log(N / df_t)$, $\quad w_{t,d} = tf_{t,d} \cdot idf_t$ (same for the query)
- $score(q, d) = \cos(\vec{q}, \vec{d})$

In [4]:
# TF of every product: pid -> {term: count / total terms of the product}
def TF(products):
    tf = {}
    for p in products:
        terms = get_terms(p)
        term_counts = Counter(terms)
        tf[p["pid"]] = {term: count / len(terms) for term, count in term_counts.items()} if terms else {}
    return tf


# IDF of every term: log(N / df)
def IDF(index, n_products):
    return {term: math.log(n_products / len(pids)) for term, pids in index.items()}


# TF-IDF weights of every product: pid -> {term: tf * idf}
def TFxIDF(tf, idf):
    return {pid: {term: value * idf[term] for term, value in term_tf.items()} for pid, term_tf in tf.items()}


# Euclidean norm of a sparse vector stored as a dict
def vector_norm(vector):
    return math.sqrt(sum(value ** 2 for value in vector.values()))


tf = TF(products)
idf = IDF(index, N)
tfidf = TFxIDF(tf, idf)
tfidf_norms = {pid: vector_norm(vector) for pid, vector in tfidf.items()}


# TF-IDF vector of the query
def query_vector(query, idf):
    terms = [term for term in preprocess_text(query) if term in idf]
    if not terms:
        return {}
    return {term: (count / len(terms)) * idf[term] for term, count in Counter(terms).items()}


# Cosine similarity between the query vector and the TF-IDF vector of product pid
def cosine_similarity(q_vector, pid):
    d_vector = tfidf[pid]
    dot_product = sum(weight * d_vector.get(term, 0.0) for term, weight in q_vector.items())
    norms = vector_norm(q_vector) * tfidf_norms[pid]
    return dot_product / norms if norms else 0.0


# AND search + TF-IDF ranking: list of (pid, score) sorted by descending cosine similarity
def search_tfidf(query):
    q_vector = query_vector(query, idf)
    # Sorted by pid so ties always keep the same order
    scores = [(pid, cosine_similarity(q_vector, pid)) for pid in sorted(search_and(query, index))]
    scores.sort(key=lambda item: item[1], reverse=True)
    return scores

### 1.2 b) BM25

$$BM25(q, d) = \sum_{t \in q} idf_t \cdot \frac{f_{t,d}\,(k_1 + 1)}{f_{t,d} + k_1\left(1 - b + b\,\dfrac{|d|}{avgdl}\right)} \qquad idf_t = \log\left(1 + \frac{N - df_t + 0.5}{df_t + 0.5}\right)$$

- $f_{t,d}$: **raw count** of the term in the product (not divided by the length, unlike the TF of TF-IDF)
- $|d|$: number of terms of the product; $avgdl$: average length in the collection (68.1 terms)
- $k_1 = 1.2$: **saturation** of the term frequency: the first occurrence of a term adds a lot, the next ones less and less
- $b = 0.75$: **partial length normalization**: long products are penalized with respect to the average length, but only partially
- We use the "+1" idf: with the classic $\log\frac{N - df_t + 0.5}{df_t + 0.5}$ a term that appears in more than half of the products gets a negative weight. Example: *cotton* appears in 71 % of the products and is a word of the test query *women printed cotton kurta*: classic idf = −0.895, so matching it would **lower** the score.

In [5]:
K1 = 1.2
B = 0.75

# Raw term counts per product, product lengths and average length (computed once)
term_counts = {p["pid"]: Counter(get_terms(p)) for p in products}
doc_lengths = {pid: sum(counts.values()) for pid, counts in term_counts.items()}
avgdl = sum(doc_lengths.values()) / len(doc_lengths)

# BM25 idf of every term: log(1 + (N - df + 0.5) / (df + 0.5)), never negative
bm25_idf = {term: math.log(1 + (N - len(pids) + 0.5) / (len(pids) + 0.5)) for term, pids in index.items()}


# BM25 score of product pid for the preprocessed query terms
def bm25_score(query_terms, pid, k1=K1, b=B):
    counts = term_counts[pid]

    # Length normalization: > 1 for products longer than average, < 1 for shorter ones
    length_norm = k1 * (1 - b + b * doc_lengths[pid] / avgdl)

    # Sum over the query terms: idf * saturated term frequency
    score = 0.0
    for term in query_terms:
        f = counts.get(term, 0)
        if f > 0:
            score += bm25_idf[term] * f * (k1 + 1) / (f + length_norm)
    return score


# AND search + BM25 ranking: list of (pid, score) sorted by descending BM25
def search_bm25(query, k1=K1, b=B):
    query_terms = set(preprocess_text(query))

    # Same candidates as TF-IDF (AND results), sorted by pid so ties keep the same order
    scores = [(pid, bm25_score(query_terms, pid, k1, b)) for pid in sorted(search_and(query, index))]
    scores.sort(key=lambda item: item[1], reverse=True)
    return scores


# Why the "+1" idf: with the classic formula a term in more than half of the products gets a negative weight
term = stemmer.stem("cotton")
df_term = len(index[term])
print(f"avgdl = {avgdl:.1f} terms per product")
print(f"'cotton' appears in {df_term / N:.0%} of the products -> idf: classic BM25 = "
      f"{math.log((N - df_term + 0.5) / (df_term + 0.5)):.3f}, BM25 used = {bm25_idf[term]:.3f}")

avgdl = 68.1 terms per product
'cotton' appears in 71% of the products -> idf: classic BM25 = -0.895, BM25 used = 0.343


### 1.3 c) Our score

Retrieval is still the AND search; our score only changes the order.

1. **Weight of each query term** (how informative the word is):

   $w_t = \log \dfrac{\text{total terms of the collection}}{\text{occurrences of } t \text{ in the collection}}$
   
   rare words (*kurta*) weigh more than common ones (*women*). Unlike the idf, it counts every occurrence of the word, not the number of products that contain it.
2. **Field-weighted presence**: $tf^{field}_{t,d} = 8 \cdot [t \in title_d] + 0.5 \cdot [t \in details_d] + 0.2 \cdot [t \in description_d]$: each field counts the term **once** (presence), so a long text that repeats a word does not win; the title is by far the most important field (a short, clean summary of what the product is), the description the least (it is long, often marketing text, and missing in 40 % of the products).
3. **Text score**: $text(q,d) = \sum_{t \in q} w_t \cdot tf^{field}_{t,d}$, divided by the maximum of the query so it is in [0, 1].
4. **Final score**: $OurScore(q,d) = \big(0.8 \cdot text_{norm}(q,d) + 0.2 \cdot \dfrac{rating_d}{5}\big) \times \begin{cases}0.5 & \text{out of stock}\\1 & \text{otherwise}\end{cases}$
5. **Duplicates**: only the best product of each (title, brand) is kept; products with the same title and a different brand are kept.

Missing rating → mean rating of the collection (neutral). Out-of-stock products are pushed down, not removed.

**Numerical fields not used:** `price`, `mrp` and `discount_pct` are parsed but left out on purpose: a cheaper or more discounted product is not more *relevant* to the query unless the user asks for it (that would be a filter or a sort option, not part of the relevance score).

In [6]:
W_TITLE, W_DETAILS, W_DESCRIPTION = 8.0, 0.5, 0.2   # importance of each field
W_TEXT, W_RATING = 0.8, 0.2                        # text relevance vs product rating
OUT_OF_STOCK_FACTOR = 0.5                          # out-of-stock products are pushed down, not removed

# Weight of every term: log(total terms of the collection / occurrences of the term in the collection)
collection_counts = Counter()
for p in products:
    collection_counts.update(get_terms(p))
total_terms = sum(collection_counts.values())
term_weight = {term: math.log(total_terms / count) for term, count in collection_counts.items()}

# Terms of each field as sets, computed once: each field counts a term only once (presence)
title_terms = {p["pid"]: set(p["title_tokens"]) for p in products}
details_terms = {p["pid"]: set(p["product_details_tokens"]) for p in products}
description_terms = {p["pid"]: set(p["description_tokens"]) for p in products}

# Mean rating of the collection, used for the products without rating
MEAN_RATING = np.mean([p["rating"] for p in products if p["rating"] is not None])


# Field-weighted text score of product pid: sum over the query terms of term weight * field presence
def text_score(query_terms, pid):
    score = 0.0
    for term in query_terms:
        field_tf = (W_TITLE * (term in title_terms[pid])
                    + W_DETAILS * (term in details_terms[pid])
                    + W_DESCRIPTION * (term in description_terms[pid]))
        score += term_weight.get(term, 0.0) * field_tf
    return score


# AND search + our score: list of (pid, score) sorted by descending score, one product per (title, brand)
def search_our_score(query):
    query_terms = set(preprocess_text(query))
    candidates = sorted(search_and(query, index))   # sorted by pid so ties always keep the same order
    if not candidates:
        return []

    # Text score of every candidate, divided by the best one of the query -> [0, 1]
    text = {pid: text_score(query_terms, pid) for pid in candidates}
    max_text = max(text.values()) or 1.0

    scores = []
    for pid in candidates:
        p = products_by_pid[pid]
        rating = p["rating"] if p["rating"] is not None else MEAN_RATING

        # Weighted sum of text and rating, then the stock penalty
        score = W_TEXT * text[pid] / max_text + W_RATING * rating / 5
        if p["out_of_stock"]:
            score *= OUT_OF_STOCK_FACTOR
        scores.append((pid, score))
    scores.sort(key=lambda item: item[1], reverse=True)

    # Duplicates: keep only the best product of each (title, brand); same title with another brand is kept
    seen = set()
    unique_scores = []
    for pid, score in scores:
        key = (tuple(products_by_pid[pid]["title_tokens"]), products_by_pid[pid]["brand"])   # ("solid", "men", "vneck", ...)
        if key not in seen:
            seen.add(key)
            unique_scores.append((pid, score))
    return unique_scores


print(f"Mean rating used for missing values: {MEAN_RATING:.2f}")

Mean rating used for missing values: 3.63


### 1.4 Comparing the rankings

All the methods rank the same AND results, so we only compare the order. In the tables each product has its position in every ranking (`-` = removed as a duplicate by our score).

#### 1.4.1 Helper functions

`show_top` prints the top-k of one method; `compare_rankings` shows, for each product, its position in every ranking.

In [7]:
RANKERS = {"tfidf": search_tfidf, "bm25": search_bm25, "our_score": search_our_score}


# Top-k of a query with the chosen method: rank, score and title
def show_top(query, method="tfidf", k=10):
    rows = []
    for position, (pid, score) in enumerate(RANKERS[method](query)[:k], start=1):
        rows.append({"rank": position, "score": round(score, 3), "title": titles[pid]})
    return pd.DataFrame(rows).set_index("rank")


# One row per product with its position in each ranking ("-" = removed from that ranking)
# top_from: methods whose top-k products are shown (default: all the compared methods)
def compare_rankings(query, methods=("tfidf", "bm25"), k=10, top_from=None):
    rankings = {method: [pid for pid, _ in RANKERS[method](query)] for method in methods}
    positions = {method: {pid: i for i, pid in enumerate(ranking, start=1)} for method, ranking in rankings.items()}

    # Products in the top-k of the chosen methods, each one only once
    top_pids = []
    for method in (top_from or methods):
        for pid in rankings[method][:k]:
            if pid not in top_pids:
                top_pids.append(pid)

    # Title, brand, stock, length and position in every ranking
    rows = []
    for pid in top_pids:
        p = products_by_pid[pid]
        row = {"title": p["title"], "brand": p["brand"], "out of stock": "yes" if p["out_of_stock"] else "",
               "len": doc_lengths[pid]}
        for method in methods:
            row[method] = positions[method].get(pid, "-")
        rows.append(row)
    return pd.DataFrame(rows)


#### 1.4.2 Our score as reference: *men black polo shirt*

Our top-10 and the position that TF-IDF and BM25 give to the same products.

In [8]:
# Our top-10 as reference: where TF-IDF and BM25 put the same products
compare_rankings("men black polo shirt", ("our_score", "tfidf", "bm25"), k=10, top_from=("our_score",))


,title,brand,out of stock,len,our_score,tfidf,bm25
0,Printed Men Polo Neck Black T-Shirt,TEE BUD,,59,1,1,2
1,"Self Design, Solid Men Polo Neck Blue, Black T-Shirt (Pack of 2)",Keo,,181,2,9,11
2,"Solid Men Polo Neck Green, Black, Yellow T-Shirt (Pack of 3)",Keo,,179,3,8,8
3,"Solid Men Polo Neck Light Blue, Green, Black T-Shirt (Pack of 3)",Keo,,181,4,6,12
4,"Solid Men Polo Neck Green, Black, Orange T-Shirt (Pack of 3)",Keo,,179,5,13,9
5,"Solid Men Polo Neck Red, Green, Black T-Shirt (Pack of 3)",Keo,,179,6,7,10
6,"Solid Men Polo Neck Green, Pink, Black T-Shirt (Pack of 3)",Keo,,181,7,14,13
7,"Solid Men Polo Neck Blue, Pink, Black T-Shirt (Pack of 3)",Keo,,181,8,11,14
8,"Solid Men Polo Neck Red, Black T-Shirt (Pack of 2)",ModeWa,,82,9,5,5
9,"Self Design, Solid Men Polo Neck Maroon, Pink, Black T-Shirt (Pac...",Keo,,183,10,18,19


#### 1.4.3 Our score as reference: *women printed cotton kurta*

In [9]:
compare_rankings("women printed cotton kurta", ("our_score", "tfidf", "bm25"), k=10, top_from=("our_score",))

,title,brand,out of stock,len,our_score,tfidf,bm25
0,Women Polka Print Cotton Blend A-line Kurta (White),FOREVER YOU,,98,1,11,9
1,Women Polka Print Cotton Blend A-line Kurta (Light Blue),FOREVER YOU,,61,2,28,17
2,Women Polka Print Cotton Blend A-line Kurta (Green),FOREVER YOU,,59,3,40,11
3,Women Printed Pure Cotton Straight Kurta (Dark Blue),allan pet,,78,4,34,19
4,Women Printed Pure Cotton Straight Kurta (Light Blue),allan pet,,78,5,36,20
5,Women Printed Pure Cotton Straight Kurta (Brown),allan pet,,80,6,20,7
6,Women Printed Cotton Blend Straight Kurta (Blue),LDHSA,,134,7,25,31
7,Women Printed Cotton Blend Straight Kurta (Beige),LDHSA,,134,8,30,32
8,Women Printed Cotton Blend Straight Kurta (Yellow),LDHSA,,134,9,29,37
9,Women Printed Cotton Blend Straight Kurta (Orange),LDHSA,,97,10,1,5


#### 1.4.4 TF-IDF vs BM25: the five test queries

The union of the two top-10 lists, with the length (`len`, number of terms) of each product.

In [10]:
for query in test_queries:
    print(f"\n{query!r}")
    display(compare_rankings(query, ("tfidf", "bm25"), k=10))


'men black polo shirt'


,title,brand,out of stock,len,tfidf,bm25
0,Printed Men Polo Neck Black T-Shirt,TEE BUD,,59,1,2
1,Printed Men Polo Neck Black T-Shirt,TEE BUD,,59,2,1
2,Checkered Men Polo Neck White T-Shirt,Steenb,,100,3,3
3,Striped Men Polo Neck Black T-Shirt,Mountain colou,,90,4,4
4,"Solid Men Polo Neck Red, Black T-Shirt (Pack of 2)",ModeWa,,82,5,5
5,"Solid Men Polo Neck Light Blue, Green, Black T-Shirt (Pack of 3)",Keo,,181,6,12
6,"Solid Men Polo Neck Red, Green, Black T-Shirt (Pack of 3)",Keo,,179,7,10
7,"Solid Men Polo Neck Green, Black, Yellow T-Shirt (Pack of 3)",Keo,,179,8,8
8,"Self Design, Solid Men Polo Neck Blue, Black T-Shirt (Pack of 2)",Keo,,181,9,11
9,"Solid Men Polo Neck Blue, Black, Orange T-Shirt (Pack of 3)",Keo,,178,10,7



'women printed cotton kurta'


,title,brand,out of stock,len,tfidf,bm25
0,Women Printed Cotton Blend Straight Kurta (Orange),LDHSA,,97,1,5
1,Women Printed Cotton Blend Straight Kurta (Orange),LDHSA,,126,2,1
2,Women Printed Cotton Blend Straight Kurta (Orange),LDHSA,,126,3,2
3,Women Printed Cotton Blend Straight Kurta (Orange),LDHSA,,126,4,3
4,Women Printed Cotton Blend Straight Kurta (Orange),LDHSA,,126,5,4
5,"Women Printed Cotton Blend Straight Kurta (Blue, Orange)",LDHSA,,122,6,6
6,Women Printed Cotton Linen Blend Straight Kurta (Blue),Mah,,63,7,15
7,Women Printed Cotton Linen Blend Straight Kurta (White),Mah,,62,8,13
8,Women Printed Cotton Blend Straight Kurta (Multicolor),LDHSA,,130,9,8
9,Women Printed Cotton Linen Blend Straight Kurta (White),Mah,,62,10,12



'grey track pants'


,title,brand,out of stock,len,tfidf,bm25
0,Striped Men Grey Track Pants,Bindass Bo,,25,1,1
1,Solid Women Grey Track Pants,saltla,,22,2,30
2,Solid Men Grey Track Pants,saltla,,22,3,29
3,Solid Men Grey Track Pants,REAZ,,34,4,15
4,"Solid Women Grey, Blue Track Pants",REAZ,,37,5,32
5,"Solid Women Grey, Blue Track Pants",REAZ,,37,6,35
6,"Solid Women Grey, Blue Track Pants",REAZ,,38,7,48
7,"Solid Women Grey, Black Track Pants",REAZ,,37,8,33
8,Solid Women Grey Track Pants,REAZ,,36,9,23
9,"Solid Men Blue, Grey Track Pants",REAZ,,38,10,46



'men denim jacket'


,title,brand,out of stock,len,tfidf,bm25
0,Full Sleeve Washed Men Denim Jacket,saltla,,42,1,4
1,Full Sleeve Washed Men Denim Jacket,saltla,yes,42,2,5
2,Full Sleeve Solid Men Denim Jacket,,,40,3,3
3,Full Sleeve Solid Men Denim Jacket,VOXA,yes,47,4,1
4,Full Sleeve Solid Men Denim Jacket,VOXA,,53,5,9
5,Full Sleeve Solid Men Denim Jacket,VOXA,,53,6,11
6,Full Sleeve Solid Men Denim Jacket,VOXA,yes,53,7,10
7,Full Sleeve Solid Men Denim Jacket,VOXA,,53,8,12
8,Full Sleeve Solid Men Denim Jacket,VOXA,yes,47,9,14
9,Full Sleeve Solid Men Jacket,VOXA,,42,10,23



'women yoga pants'


,title,brand,out of stock,len,tfidf,bm25
0,Solid Women Black Track Pants,Pu,yes,129,1,4
1,Solid Women Black Track Pants,Onei,,76,2,1
2,"Solid Women Black, Dark Blue Track Pants",Onei,,81,3,2
3,"Solid Women Black, Dark Blue Track Pants",Onei,,81,4,3
4,Solid Women Black Track Pants,Onei,,81,5,7
5,Solid Women Dark Blue Track Pants,Onei,,83,6,8
6,Solid Women Grey Track Pants,Onei,,81,7,6
7,Solid Women Blue Track Pants,SMART ATTRACTI,,79,8,5
8,Solid Women Black Track Pants,VC-Vanor,,147,9,9
9,Solid Women Black Track Pants,VC-Vanor,,147,10,10


#### 1.4.5 Summary of the top-10 of each method

For each query and method: mean length of the top-10 (and of all the AND results, for reference), out-of-stock products, duplicates (same title and brand) and mean rating.

In [11]:
# Summary of the top-k of every method: mean length, out-of-stock products, duplicates and mean rating
def ranking_summary(queries, methods=("tfidf", "bm25", "our_score"), k=10):
    rows = []
    for query in queries:
        and_mean_len = np.mean([doc_lengths[pid] for pid in search_and(query, index)])
        for method in methods:
            top = [products_by_pid[pid] for pid, _ in RANKERS[method](query)[:k]]
            keys = [(tuple(p["title_tokens"]), p["brand"]) for p in top]
            ratings = [p["rating"] for p in top if p["rating"] is not None]
            rows.append({
                "query": query,
                "method": method,
                "mean len top-k": round(np.mean([doc_lengths[p["pid"]] for p in top]), 1),
                "mean len AND": round(and_mean_len, 1),
                "out of stock": sum(p["out_of_stock"] for p in top),
                "duplicates": len(top) - len(set(keys)),
                "mean rating": round(np.mean(ratings), 2) if ratings else None,
            })
    return pd.DataFrame(rows).set_index(["query", "method"])


ranking_summary(test_queries)

mean len top-k  mean len AND  \
query                      method                                    
men black polo shirt       tfidf               128.8         138.6   
                           bm25                120.2         138.6   
                           our_score           158.5         138.6   
women printed cotton kurta tfidf               104.0          90.8   
                           bm25                110.5          90.8   
                           our_score            95.3          90.8   
grey track pants           tfidf                32.6          70.1   
                           bm25                 48.0          70.1   
                           our_score            73.1          70.1   
men denim jacket           tfidf                47.2          99.0   
                           bm25                 61.3          99.0   
                           our_score            72.7          99.0   
women yoga pants           tfidf                98.5         121.9   
                           bm25                 98.5         121.9   
                           our_score           128.1         121.9   

                                      out of stock  duplicates  mean rating  
query                      method                                            
men black polo shirt       tfidf                 0           1         3.68  
                           bm25                  0           1         3.66  
                           our_score             0           0         3.93  
women printed cotton kurta tfidf                 0           5         4.01  
                           bm25                  0           4         4.00  
                           our_score             0           0         4.22  
grey track pants           tfidf                 0           2         3.56  
                           bm25                  1           3         2.78  
                           our_score             0           0         4.32  
men denim jacket           tfidf                 4           6         3.58  
                           bm25                  5           4         3.33  
                           our_score             0           0         3.58  
women yoga pants           tfidf                 1           3         3.92  
                           bm25                  1           3         3.92  
                           our_score             0           0         3.78

### 1.5 Analysis

The full discussion is in the report. Main points from the tables above:

- TF-IDF prefers short products (in *grey track pants* its top-10 has 32.6 terms on average, the AND results 70.1), because it divides by the length twice (tf and cosine).
- BM25 normalizes the length only partially and saturates the term frequency, so longer products that repeat the query words go up (Fairdea products: TF-IDF 19-23, BM25 5-9).
- Neither looks at stock or duplicates. Our score has 0 out-of-stock products and 0 duplicates in the top-10 for every query, and the best mean rating in 4 of the 5 queries.

## 2. word2vec + cosine similarity

We train our own word2vec model on the products (one sentence per product, same tokens as the index, so query words and model words match). A text is the mean of the vectors of its words that are in the vocabulary, and the AND results are ranked by the cosine between the query vector and each product vector.

### 2.1 Train word2vec

In [12]:
from gensim.models import Word2Vec
import zlib


# Deterministic hash for gensim: Python's hash() changes between runs and would change the model every time
def stable_hash(text):
    return zlib.crc32(str(text).encode("utf-8"))


# 2.1: one "sentence" per product (its tokens), then train the model
sentences = [get_terms(product) for product in products]
w2v = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=2, sg=1, epochs=10, seed=42, workers=1, hashfxn=stable_hash)


# sanity check of the model, e.g. the nearest words of some query terms

for word in ["kurta", "denim", "shirt"]:
    token = preprocess_text(word)[0]

    if token in w2v.wv:
        print(f"\nWords similar to '{word}':")
        print(w2v.wv.most_similar(token, topn=5))


Words similar to 'kurta':
[('pyjama', 0.6287217736244202), ('churidar', 0.6139382123947144), ('ethnic', 0.5801189541816711), ('ldhsati', 0.5480777025222778), ('pajama', 0.5345406532287598)]

Words similar to 'denim':
[('jean', 0.5815389752388), ('botton', 0.5480282306671143), ('talk', 0.5366770029067993), ('classiest', 0.5342609286308289), ('sr6617', 0.528221607208252)]

Words similar to 'shirt':
[('atm', 0.6492501497268677), ('shirtr', 0.5731101036071777), ('m18', 0.5625478625297546), ('oxford', 0.5380017757415771), ('sorang', 0.5363722443580627)]


### 2.2 Text vector (average of the word vectors)

In [13]:
# Average of the word2vec vectors of the tokens that are in the vocabulary (zero vector if none is)
def text_vector(tokens):
    vectors = [w2v.wv[token] for token in tokens if token in w2v.wv]

    if not vectors:
        return np.zeros(w2v.vector_size, dtype=np.float32)

    return np.mean(vectors, axis=0)


# Cosine similarity between two dense vectors (0 if one of them is the zero vector)
def dense_cosine(a, b):
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return float(np.dot(a, b) / (norm_a * norm_b))


# 2.2: vector of every product, computed once -> {pid: vector}
w2v_doc_vectors = {product["pid"]: text_vector(get_terms(product)) for product in products}

### 2.3 word2vec ranking

In [14]:
# AND search + word2vec ranking: list of (pid, score) sorted by descending cosine similarity
def search_word2vec(query):
    matching_pids = search_and(query, index)

    if not matching_pids:
        return []

    query_tokens = preprocess_text(query)
    q_vector = text_vector(query_tokens)

    scores = []

    for pid in matching_pids:
        d_vector = w2v_doc_vectors[pid]
        score = dense_cosine(q_vector, d_vector)

        scores.append((pid, score))

    scores.sort(key=lambda item: (-item[1], item[0]))

    return scores


RANKERS["word2vec"] = search_word2vec

### 2.4 Top-20 for the five test queries

*women yoga pants* has only 19 AND results, so its list has 19 products.

In [15]:
# 2.4: top-20 of each test query with word2vec
for query in test_queries:
    print(f"\n{query!r}")
    display(show_top(query, "word2vec", k=20))


'men black polo shirt'


,score,title
rank,,
1,0.767,Striped Men Polo Neck Black T-Shirt
2,0.764,Checkered Men Polo Neck White T-Shirt
3,0.743,Tie & Dye Men Polo Neck Brown T-Shirt
4,0.739,Men Regular Fit Solid Mandarin Collar Casual Shirt
5,0.739,Men Regular Fit Solid Slim Collar Casual Shirt
6,0.739,Men Regular Fit Washed Mandarin Collar Casual Shirt
7,0.738,Men Regular Fit Solid Spread Collar Casual Shirt
8,0.737,Men Regular Fit Solid Spread Collar Casual Shirt
9,0.736,"Solid Men Polo Neck Red, Black T-Shirt (Pack of 2)"



'women printed cotton kurta'


,score,title
rank,,
1,0.832,Women Printed Cotton Blend Straight Kurta (Orange)
2,0.832,Women Printed Cotton Blend Straight Kurta (Orange)
3,0.832,Women Printed Cotton Blend Straight Kurta (Orange)
4,0.832,Women Printed Cotton Blend Straight Kurta (Orange)
5,0.821,"Women Printed Cotton Blend Straight Kurta (Blue, Orange)"
6,0.813,Women Printed Cotton Blend Straight Kurta (Multicolor)
7,0.804,Women Kurta and Pyjama Set Cotton Blend
8,0.804,Women Kurta and Pyjama Set Cotton Blend
9,0.803,Women Kurta and Pyjama Set Cotton Blend



'grey track pants'


,score,title
rank,,
1,0.817,"Solid Women Black, Grey Track Pants"
2,0.816,"Solid Men Black, Grey Track Pants"
3,0.816,"Solid Men Black, Grey Track Pants"
4,0.816,"Solid Men Black, Grey Track Pants"
5,0.815,Solid Women Grey Track Pants
6,0.815,Solid Women Grey Track Pants
7,0.814,Solid Men Grey Track Pants
8,0.814,Solid Men Grey Track Pants
9,0.814,Solid Men Grey Track Pants



'men denim jacket'


,score,title
rank,,
1,0.795,Full Sleeve Printed Men Denim Casual Jacket
2,0.782,Full Sleeve Solid Men Denim Jacket
3,0.776,Full Sleeve Washed Men Denim Jacket
4,0.775,Full Sleeve Washed Men Denim Jacket
5,0.760,Full Sleeve Solid Men Denim Jacket
6,0.759,Full Sleeve Solid Men Denim Jacket
7,0.758,Full Sleeve Solid Men Denim Jacket
8,0.752,Full Sleeve Solid Men Denim Jacket
9,0.750,Full Sleeve Solid Men Denim Jacket



'women yoga pants'


,score,title
rank,,
1,0.772,Solid Women Black Track Pants
2,0.762,"Solid Women Black, Dark Blue Track Pants"
3,0.762,"Solid Women Black, Dark Blue Track Pants"
4,0.758,Solid Women Black Track Pants
5,0.747,Solid Women Dark Blue Track Pants
6,0.745,Solid Women Grey Track Pants
7,0.744,Solid Women Black Track Pants
8,0.740,Solid Women Dark Blue Track Pants
9,0.739,Solid Women Grey Track Pants


### 2.5 word2vec compared with the other rankings

How many products of the word2vec top-20 are also in the top-20 of each method, and the summary of 1.4.5 for the word2vec top-10.

In [16]:
# How many products of the top-k of `reference` are also in the top-k of each of the other methods
def top_overlap(queries, reference="word2vec", others=("tfidf", "bm25", "our_score"), k=20):
    rows = []
    for query in queries:
        ref = {pid for pid, _ in RANKERS[reference](query)[:k]}
        row = {"query": query, "AND results": len(search_and(query, index))}
        for method in others:
            row[method] = len(ref & {pid for pid, _ in RANKERS[method](query)[:k]})
        rows.append(row)
    return pd.DataFrame(rows).set_index("query")


display(top_overlap(test_queries))

# Same summary as 1.4.5 for the word2vec top-10 (duplicates, stock and rating are not taken into account)
display(ranking_summary(test_queries, methods=("word2vec",)))

# Where does "yoga" appear in the 19 AND results of "women yoga pants"?
yoga = stemmer.stem("yoga")
yoga_pids = search_and("women yoga pants", index)
fields = {"title": title_terms, "details": details_terms, "description": description_terms}
print("Products with 'yoga' in each field:", {name: sum(yoga in terms[pid] for pid in yoga_pids) for name, terms in fields.items()})
print(f"word2vec vocabulary: {len(w2v.wv):,} words")

,AND results,tfidf,bm25,our_score
query,,,,
men black polo shirt,29,12,14,13
women printed cotton kurta,42,12,10,9
grey track pants,327,7,15,0
men denim jacket,73,14,15,6
women yoga pants,19,19,19,11


,,mean len top-k,mean len AND,out of stock,duplicates,mean rating
query,method,,,,,
men black polo shirt,word2vec,97.7,138.6,0,2,3.12
women printed cotton kurta,word2vec,100.1,90.8,0,5,3.63
grey track pants,word2vec,59.1,70.1,0,5,2.69
men denim jacket,word2vec,60.4,99.0,4,7,3.54
women yoga pants,word2vec,102.1,121.9,1,5,4.15


Products with 'yoga' in each field: {'title': 0, 'details': 1, 'description': 18}
word2vec vocabulary: 7,831 words


### 2.6 Observations

- The scores are very close (about 0.70-0.83): the mean of 60-150 word vectors looks similar for all the products of a category.
- word2vec only compares text, so duplicates come back and rating and stock are ignored (0 products in common with our top-20 for *grey track pants*).
- *yoga* is in the title of none of the 19 *women yoga pants* results, they are all track pants.
- Since the AND search already removed the products without the exact query words, word2vec can only reorder them.

## 3. A better representation than word2vec?

Answered in the report: the limits of averaging word vectors, and the pros and cons of Doc2Vec, Sentence2Vec and Sentence-BERT.